In [1]:
import os, sys, numpy as np, pandas as pd
# 若新 notebook 不在 scripts/，把 scripts 路徑加進來：
# sys.path.append(r"c:\...\MS2_Data_PyTorch\scripts")
from BPM.BPM import predict, score_m10, score_m35
from BPM.util import Motif2Seqs


In [2]:
ALL_HEX = list(Motif2Seqs('N'*6))   # 4096 種 6-mer
SPACER  = 'A'*17                    # spacer 只看「長度=17」，內容不影響分數
REF_M35 = 'TTGACA'                  # 掃 -10 時固定的參考 -35 (consensus)
REF_M10 = 'TATAAT'                  # 掃 -35 時固定的參考 -10 (consensus)

# 變動 -10（固定 consensus -35 + 17 spacer）
df10 = pd.DataFrame({'m10': ALL_HEX})
df10['LogExp'] = [predict(REF_M35 + SPACER + h) for h in ALL_HEX]

# 變動 -35（固定 consensus -10）
df35 = pd.DataFrame({'m35': ALL_HEX})
df35['LogExp'] = [predict(h + SPACER + REF_M10) for h in ALL_HEX]


In [28]:
def pick_strength_levels(df, seqcol, strength_levels=(20, 40, 60, 80)):
    d = df.sort_values('LogExp', ascending=False).reset_index(drop=True)  # 強→弱
    N = len(d)

    rows = []
    for pct in strength_levels:
        t = 1 - pct / 100
        idx = int(round(t * (N - 1)))
        row = d.iloc[idx].copy()
        row['strength_pct'] = pct
        rows.append(row)

    out = pd.DataFrame(rows).reset_index(drop=True)
    return out[[seqcol, 'LogExp', 'strength_pct']]
print("=== -10 20/40/60/80 強度代表 ===")
display(pick_strength_levels(df10, 'm10', strength_levels=(20, 40, 60, 80)))

print("=== -35 20/40/60/80 強度代表 ===")
display(pick_strength_levels(df35, 'm35', strength_levels=(20, 40, 60, 80)))

=== -10 20/40/60/80 強度代表 ===


,m10,LogExp,strength_pct
0,CTTCGG,0.555821,20
1,AGTTCC,0.682063,40
2,GTACCT,0.987263,60
3,AAGTAA,1.566462,80


=== -35 20/40/60/80 強度代表 ===


,m35,LogExp,strength_pct
0,CCTTTG,1.131238,20
1,CTAGCC,1.505044,40
2,AAGTAG,1.891244,60
3,TCCCCG,2.352189,80


In [ ]:
def pick_levels(df, seqcol, n_levels=5):
    d = df.sort_values('LogExp', ascending=False).reset_index(drop=True)  # 強→弱
    N = len(d)
    targets = [(i + 0.5) / n_levels for i in range(n_levels)]   # 區間中心
    out = pd.DataFrame([d.iloc[int(round(t*(N-1)))] for t in targets]).reset_index(drop=True)
    out['strength_pct'] = [round((1 - t) * 100) for t in targets]  # 100=最強
    return out[[seqcol, 'LogExp', 'strength_pct']]

print("=== -10 各強度代表 ===")
display(pick_levels(df10, 'm10', n_levels=5))
print("=== -35 各強度代表 ===")
display(pick_levels(df35, 'm35', n_levels=5))


=== -10 各強度代表 ===


,m10,LogExp,strength_pct
0,TATGTG,2.065411,190
1,TTAGAC,1.233570,170
2,TGCGGA,0.803521,150
3,CGCTCC,0.600065,130
4,GGGGGA,0.524248,110


=== -35 各強度代表 ===


,m35,LogExp,strength_pct
0,TGTTTA,2.638414,190
1,ATTACT,2.112106,170
2,CTAATC,1.693999,150
3,ATACGC,1.319381,130
4,CCATCG,0.912580,110


In [6]:
from BPM.BPM import score_promoter, score_promoter_elements

def strength_pct(value, ref_series):
    """value 在 ref 分布中的強度百分位 (100=最強)。"""
    return (ref_series < value).mean() * 100

def score_seq(seqs):
    """輸入完整 promoter (m35 + spacer + m10)，回傳能量/GFP/各元件強度百分位。"""
    if isinstance(seqs, str):
        seqs = [seqs]
    rows = []
    for s in seqs:
        s = s.upper()
        m35, m10, spacer = s[:6], s[-6:], s[6:-6]
        e35, esp, e10 = score_promoter_elements(s)     # 各元件能量 (dG)
        logexp = predict(s)                            # log10 表現量
        # 元件強度百分位（放在標準 consensus 背景下比較，才可橫向比較）
        m10_pct = strength_pct(predict(REF_M35 + SPACER + m10), df10['LogExp'])
        m35_pct = strength_pct(predict(m35 + SPACER + REF_M10), df35['LogExp'])
        rows.append({
            'Sequence': s, 'm35': m35, 'spacer_len': len(spacer), 'm10': m10,
            'E_total': e35 + esp + e10,
            'E_m35': e35, 'E_spacer': esp, 'E_m10': e10,
            'LogGFP': logexp, 'GFP': 10**logexp,
            'm35_pct': m35_pct, 'm10_pct': m10_pct,
        })
    return pd.DataFrame(rows)

# === 範例 ===
print(score_seq("TCCAGT" + "T"*17 + "CAAGAT"))
print(score_seq("TCGTCA" + "T"*17 + "CACGCT"))
print(score_seq("TTGACA" + "T"*17 + "TATAAT"))  # consensus

# 多條：
# print(score_seq(["TTGACAaaaaaaaaaaaaaaaaaTATAAT", "TTTACAaaaaaaaaaaaaaaaaaTACAAT"]))


                        Sequence     m35  spacer_len     m10   E_total  \
0  TCCAGTTTTTTTTTTTTTTTTTTCAAGAT  TCCAGT          17  CAAGAT -2.656949   

      E_m35  E_spacer     E_m10  LogGFP       GFP    m35_pct    m10_pct  
0  0.059492         0 -2.716441  0.5637  3.661844  50.805664  94.921875  
                        Sequence     m35  spacer_len     m10   E_total  \
0  TCGTCATTTTTTTTTTTTTTTTTCACGCT  TCGTCA          17  CACGCT -4.622239   

      E_m35  E_spacer     E_m10    LogGFP        GFP    m35_pct    m10_pct  
0 -2.277376         0 -2.344863  1.202698  15.947685  96.118164  91.967773  
                        Sequence     m35  spacer_len     m10   E_total  \
0  TTGACATTTTTTTTTTTTTTTTTTATAAT  TTGACA          17  TATAAT -9.457412   

      E_m35  E_spacer     E_m10    LogGFP         GFP    m35_pct    m10_pct  
0 -3.937519         0 -5.519893  2.987234  971.034011  99.975586  99.975586  


In [5]:
# === 寫入共用元件 CSV（-35 / -10，由 BPM 評分）===
# 共用寬表：每支元件腳本只寫/取代自己負責的欄，不動別人的欄
SHARED_CSV = "../tables/elements_shared.csv"

def write_column(csv_path, coldata):
    """coldata = {欄名: [值,...]}；自動對齊長度、保留別人寫的欄。"""
    base = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()
    new = pd.DataFrame(coldata)
    n = max(len(base), len(new))
    base = base.reindex(range(n)); new = new.reindex(range(n))
    for c in new.columns:
        base[c] = new[c]
    base.to_csv(csv_path, index=False)
    return base

# 各挑 5 條（強→弱區間中心）
picks35 = pick_levels(df35, 'm35', n_levels=5)
picks10 = pick_levels(df10, 'm10', n_levels=5)

write_column(SHARED_CSV, {
    'm35_seq': picks35['m35'].tolist(),
    'm35_pct': picks35['strength_pct'].tolist(),
    'm10_seq': picks10['m10'].tolist(),
    'm10_pct': picks10['strength_pct'].tolist(),
})
print("✅ 已寫入 m35 / m10 →", SHARED_CSV)
pd.read_csv(SHARED_CSV)


✅ 已寫入 m35 / m10 → ../tables/elements_shared.csv


,m35_seq,m35_pct,m10_seq,m10_pct,DIS_seq,DIS_pct,spacer_seq,spacer_pct,UP_seq,UP_pct,ITS_seq,ITS_pct
0,TGTTTA,90,TATGTG,90,TTGAAGCT,90,TACGTACTTTGCATCGA,88.0,CTTTGTTATAGTAACTGTG,90,CTGTTCAGAG,90
1,ATTACT,70,TTAGAC,70,AGAGCTTG,70,AAATATCAGGTTGCCAC,62.0,ATGACATACACCCTGTTCA,70,CAGATTTCTT,70
2,CTAATC,50,TGCGGA,50,TACGCCTT,50,TAATTCAACTAGGTCTG,NaN,AATCATGCAGGGTACGTTC,50,ATATCAGACC,50
3,ATACGC,30,CGCTCC,30,GAAGCTCG,30,TAATTCAACTAGGTCCA,38.0,AGCTATCATATGGGTTGAC,30,TTGACCTGCA,30
4,CCATCG,10,GGGGGA,10,CCTGCCGA,10,CACTATTGCTATGATCC,12.0,TACCCTAACCTAATCGGTT,10,ATCGGGCTGG,10
